In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "custseg").exists():   # project root = folder that holds the package
    if ROOT == ROOT.parent:
        raise RuntimeError("run the notebook from inside the project folder")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option("display.max_columns", 40); pd.set_option("display.width", 200)
from custseg import config, db
img = lambda p, w=900: display(Image(filename=str(ROOT / p), width=w))

# 02 · Experiment 1 — RFM rule-based segmentation

In [2]:
from custseg.segmentation import RFM_RULES
pd.DataFrame([(n, d) for n, d, _ in RFM_RULES], columns=['segment','rule (first match wins)'])

,segment,rule (first match wins)
0,Champions,"repeat buyer (F>=2), recent (R>=4), high spend..."
1,Potential Loyalists,"repeat buyer (F>=2), recent (R>=4), lower spend"
2,Loyal Customers,"repeat buyer (F>=2), mid recency (R=3)"
3,At Risk,lapsing (R<=2) but previously valuable: repeat...
4,Big Spenders,"one purchase, not lapsed (R>=3), top-quintile ..."
5,New Customers,one purchase in the most recent quintile (R=5)
6,Promising,"one purchase, recent (R=4)"
7,Need Attention,"one purchase, mid recency (R=3)"
8,Hibernating,"one purchase, low recency (R=2), low/mid spend"
9,Lost,"one purchase, oldest quintile (R=1), low/mid s..."


In [3]:
seg = pd.read_csv(ROOT / '02_data/processed/customer_segmentation.csv')
t = seg.groupby('rfm_segment').agg(customers=('customer_key','size'), revenue=('historical_clv','sum'))
t['customer_share'] = t.customers / t.customers.sum(); t['revenue_share'] = t.revenue / t.revenue.sum()
t.sort_values('revenue', ascending=False).round(3)

,customers,revenue,customer_share,revenue_share
rfm_segment,,,,
At Risk,14688,4504352.33,0.158,0.293
Big Spenders,10147,4438624.49,0.109,0.289
Promising,14594,1414213.44,0.157,0.092
Need Attention,14829,1410565.77,0.159,0.092
New Customers,14600,1405813.98,0.157,0.091
Lost,11351,822753.64,0.122,0.054
Hibernating,11096,808740.92,0.119,0.053
Champions,978,365267.14,0.011,0.024
Loyal Customers,602,181570.92,0.006,0.012


In [4]:
log = pd.read_csv(ROOT / '07_customer_segmentation/results/experiment_log.csv')
log[log.experiment_id == 'E1-rfm-rules'].T

,0
experiment_id,E1-rfm-rules
experiment,E1 RFM rule-based
dataset_version,ca930db9b5cf
feature_configuration,rfm_scores
features,"R, F, M scores"
preprocessing,quintile scoring
algorithm,rule-based (quintile scoring)
hyperparameters,"{""r_bins"": ""quintiles"", ""m_bins"": ""quintiles"",..."
n_clusters,10
noise_share,0.0
